# Пересечение аудитории каналов
Пример расчета пересечения накопленного охвата телекомпаний

## Описание задачи и условий расчета

- Россия 0+
- Период: 12.12.2022 - 18.12.2022
- Временной интервал: 05:00-29:00
- ЦА: Все 18+
- Место просмотра: Все места (Дом+Дача)
- Каналы: Первый канал, Россия 1, НТВ, РЕН ТВ										
- Статистики: Накопленный охват в 000 и %

## Инициализация

При построении отчета первый шаг в любом ноутбуке - загрузка библиотек, которые помогут обращаться к TVI API и работать с данными.

Выполните следующую ячейку, для этого перейдите в нее и нажмите Ctrl+Enter

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import re
import json
import datetime
import time
import pandas as pd
#import matplotlib.pyplot as plt
from IPython.display import JSON

from mediascope_api.core import utils
from mediascope_api.core import net as mscore
from mediascope_api.mediavortex import tasks as cwt
from mediascope_api.mediavortex import catalogs as cwc

# Настраиваем отображение

# Включаем отображение всех колонок
pd.set_option('display.max_columns', None)
# Задаем максимальное количество выводимых строк. Раскомментируйте нужную строку
# 200 строк
# pd.set_option("display.max_rows", 200)
# Отображаем все строки. ВАЖНО! Отображение большого DataFrame требует много ресурсов
# pd.set_option("display.max_rows", None)

# Cоздаем объекты для работы с TVI API
mnet = mscore.MediascopeApiNetwork()
mtask = cwt.MediaVortexTask()
cats = cwc.MediaVortexCats()

## Справочники

Получим идентификаторы, которые будут использоваться для формирования условий расчета

In [ ]:
# Национальные телекомпании Первый канал, Россия 1, НТВ, РЕН ТВ
cats.get_tv_net(name=['Первый канал','Россия 1', 'НТВ', 'РЕН ТВ'])

## Формирование задания
Зададим условия расчета

In [ ]:
# Период указывается в виде списка ('Начало', 'Конец') в формате 'YYYY-MM-DD'. 
date_filter = [('2022-12-12', '2022-12-18')]

# Задаем дни недели
weekday_filter = None

# Задаем тип дня
daytype_filter = None

# Задаем временной интервал для основной оси
time_filter = 'timeBand1 >= 50000 AND timeBand1 < 290000' 

# Задаем временной интервал для оси пересечения
duplication_time_filter = 'timeBand1 >= 50000 AND timeBand1 < 290000'

# Задаем ЦА: Все 18+
basedemo_filter = 'age >= 18'

# Задаем место просмотра
location_filter = None

# Задаем каналы для основной оси: национальные телекомпании Первый канал, Россия 1, НТВ, РЕН ТВ
company_filter = 'tvNetId IN (1, 2, 4, 60)'

# Задаем каналы для оси пересечения: национальные телекомпании Первый канал, Россия 1, НТВ, РЕН ТВ
duplication_company_filter = 'tvNetId IN (1, 2, 4, 60)'

# Указываем список статистик для расчета (для расчета среднесуточного охвата используйте DuplAvReach000, DuplAvReachPer)
statistics = [
    'CumDupl000',
    'CumDuplPer'
]

# Указываем срезы: телесети
slices = [
    'tvNetName',
    'duplicationTvNetName',
]

# Задаем условия сортировки
sortings = None

# Задаем опции расчета
options = {
    "kitId": 1, # набор данных TV Index Russia all
    "useNbd": False, # nbd коррекция выключена
}

## Расчет задания

In [ ]:
# Формируем задание для API TV Index в формате JSON
task_json = mtask.build_duplication_timeband_task('duplication-timeband', date_filter=date_filter,
                                                  weekday_filter=weekday_filter, daytype_filter=daytype_filter, 
                                                  basedemo_filter=basedemo_filter,
                                                  time_filter=time_filter, company_filter=company_filter,
                                                  duplication_time_filter=duplication_time_filter,
                                                  duplication_company_filter=duplication_company_filter,
                                                  location_filter=location_filter,
                                                  slices=slices, statistics=statistics, options=options)

# Отправляем задание на расчет и ждем выполнения
task_dup_timeband = mtask.wait_task(mtask.send_duplication_timeband_task(task_json))

# Получаем результат
df = mtask.result2table(mtask.get_result(task_dup_timeband))

# Преобразуем столбцы с временными интервалами в нужный формат
df['time'] = df['time'].apply(utils.convert_time_condition)
df['duplicationTime'] = df['duplicationTime'].apply(utils.convert_time_condition)

df

## Настройка внешнего вида таблицы
Пропустите этот шаг, если хотите экспортировать таблицу в ее текущем виде

In [ ]:
# Формируем сводную таблицу: 
# в строках - национальная телекомпания и временной интервал основной оси; 
# в столбцах - национальная телекомпания и временной интервал оси пересечения;
# в значениях - статистики 

df = df.pivot_table(index=['tvNetName','time'], 
                        columns=['duplicationTvNetName','duplicationTime'], 
                        values=statistics, 
                        aggfunc='first')

df

## Экспорт в Excel
По умолчанию файл сохраняется в папку `excel`

In [ ]:
df_info = mtask.task_builder.get_report_info()

with pd.ExcelWriter(mtask.task_builder.get_excel_filename('russia_duplication_timeband_01_duplication_reach')) as writer:
    df.to_excel(writer, sheet_name='Report', index=True, merge_cells=False)
    df_info.to_excel(writer, sheet_name='Info', index=False)